# SENSERO – Export patch centres to per-tile Parquet files

Converts a **point Shapefile** (created in QGIS) into one Parquet file per
Sentinel-2 tile. These Parquet files are consumed by the downstream patch
extractor code.

## Context

The input Shapefile contains one point per patch centre, each carrying a
`top_layer` attribute that identifies the source Sentinel-2 granule
(e.g. `S2A_MSIL2A_20180928T090731_N0500_R050_T35TMN`). The script groups
points by tile name and writes a compact Parquet file for each tile with
columns:

| Column | Description |
|--------|-------------|
| `FILENAME` | Sentinel-2 granule identifier (same for all rows in the file) |
| `LONGITUDE` | Centre longitude (EPSG:4326) |
| `LATITUDE` | Centre latitude (EPSG:4326) |

## Requirements

```
pip install geopandas pandas pyarrow tqdm
```


## Configuration

In [ ]:
import os
import re

import geopandas as gpd
import pandas as pd
from tqdm import tqdm

# ── CONFIG ─────────────────────────────────────────────────────────────
IN_SHP     = "/home/ubuntu/SENSERO/patch_centers.shp"          # Input Shapefile (points with top_layer field)
OUT_DIR    = "/home/ubuntu/SENSERO/patch_centers_parquet"      # Output folder for per-tile Parquet files
ENGINE     = "pyarrow"       # Parquet engine: "pyarrow" or "fastparquet"
COMPR      = None            # Compression: "snappy", "gzip", or None
TARGET_CRS = "EPSG:4326"    # Coordinates are reprojected to this CRS
OVERWRITE  = True            # If False, skip tiles whose Parquet already exists


## Helpers

In [ ]:
def sanitize(name: str) -> str:
    """Replace characters unsafe for filenames with underscores."""
    return re.sub(r'[^A-Za-z0-9._\-]+', '_', str(name))


def col_ci(df, wanted: str) -> str:
    """Case-insensitive column lookup. Raises KeyError if not found."""
    wl = wanted.lower()
    for c in df.columns:
        if c.lower() == wl:
            return c
    raise KeyError(f"Column '{wanted}' not found (case-insensitive).")


## Load and prepare

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

gdf = gpd.read_file(IN_SHP)
if gdf.empty:
    raise RuntimeError("Input Shapefile has no features.")

print(f"Loaded : {len(gdf):,} features, CRS = {gdf.crs}")

# Locate the tile-name column (case-insensitive)
tl = col_ci(gdf, "top_layer")

if gdf.crs is None:
    raise ValueError("Input Shapefile has no CRS; define it in QGIS before exporting.")

# Reproject to WGS-84 if needed
if str(gdf.crs) != TARGET_CRS:
    gdf = gdf.to_crs(TARGET_CRS)
    print(f"Reprojected to {TARGET_CRS}")

# Explode MultiPoints (unlikely but safe)
if (gdf.geometry.geom_type == "MultiPoint").any():
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    print(f"Exploded MultiPoints → {len(gdf):,} features")

# Extract coordinates
gdf["LONGITUDE"] = gdf.geometry.x
gdf["LATITUDE"]  = gdf.geometry.y

# Drop rows with invalid coordinates
before = len(gdf)
gdf = gdf[
    pd.to_numeric(gdf["LONGITUDE"], errors="coerce").notna() &
    pd.to_numeric(gdf["LATITUDE"],  errors="coerce").notna()
].copy()
if len(gdf) < before:
    print(f"Dropped {before - len(gdf)} features with invalid coordinates")


## Write per-tile Parquet files

In [ ]:
tiles = gdf[tl].astype(str)
groups = gdf.groupby(tiles, dropna=True)

total_points = 0
skipped = 0

for tile, sub in tqdm(groups, desc="Writing Parquet per tile"):
    out_pq = os.path.join(OUT_DIR, f"{sanitize(tile)}.parquet")

    if not OVERWRITE and os.path.exists(out_pq):
        skipped += 1
        continue

    df_out = pd.DataFrame({
        "FILENAME":  [tile] * len(sub),
        "LONGITUDE": sub["LONGITUDE"].to_numpy(),
        "LATITUDE":  sub["LATITUDE"].to_numpy(),
    })

    df_out.to_parquet(
        out_pq,
        index=False,
        engine=ENGINE,
        compression=COMPR if ENGINE == "pyarrow" else None,
    )
    total_points += len(df_out)

print(f"\nTiles written : {groups.ngroups - skipped}")
if skipped:
    print(f"Tiles skipped : {skipped} (already exist, OVERWRITE=False)")
print(f"Total points  : {total_points:,}")
print(f"Output folder : {OUT_DIR}")
